# RetainIQ — Phase 8.3: Benchmark Gap Analysis & Management Views

## Objective

I turn benchmark gaps into a compact management-analysis layer. I focus on three questions:

1. Which segments are materially above or below the portfolio churn benchmark?
2. Which markets are above their peer benchmark while also showing meaningful historical revenue exposure?
3. How do Phase 7 retention-priority entities sit relative to benchmark conditions?

I use neutral benchmark language such as **above**, **within**, and **below** rather than inventing an absolute performance score.

### Notebook presentation rule

For every relevant analytical code cell, I place a **Result & conclusion** markdown cell immediately after it. Setup-only cells are not padded with artificial conclusions.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

ROOT = Path("..")
OUTPUT_DIR = ROOT / "outputs"
CONFIG_DIR = ROOT / "config"

def print_result(message):
    print(f"Result: {message}")

In [2]:
segment_benchmark = pd.read_csv(OUTPUT_DIR / "segment_benchmark_gap.csv")
state_benchmark = pd.read_csv(OUTPUT_DIR / "state_benchmark_gap.csv")
city_benchmark = pd.read_csv(OUTPUT_DIR / "city_benchmark_gap.csv")
segment_priority = pd.read_csv(OUTPUT_DIR / "segment_priority.csv")
city_priority = pd.read_csv(OUTPUT_DIR / "city_priority.csv")

for frame in [segment_benchmark, state_benchmark, city_benchmark, segment_priority, city_priority]:
    for col in ["customers", "churn_rate_pct", "avg_cltv", "revenue_at_risk", "priority_score", "churn_gap_pp", "churn_gap_to_peer_city_pp"]:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")

params = pd.read_csv(CONFIG_DIR / "benchmark_parameters.csv")
tolerance_pp = float(params.loc[params["parameter"] == "benchmark_tolerance_pp", "value"].iloc[0])

print_result("Loaded the benchmark-gap and Phase 7 prioritization outputs for management analysis.")

Result: Loaded the benchmark-gap and Phase 7 prioritization outputs for management analysis.


### Result & conclusion

The benchmark-gap outputs can now be combined with Phase 7 strategy priorities without recalculating the underlying metrics.

In [3]:
segment_summary = (
    segment_benchmark.groupby("benchmark_status", dropna=False)
    .agg(
        segments=("segment_id", "count"),
        customers=("customers", "sum"),
        churned_customers=("churned_customers", "sum"),
        historical_revenue_at_risk=("revenue_at_risk", "sum"),
    )
    .reset_index()
)
segment_summary["customer_share_pct"] = np.where(
    segment_summary["customers"].sum() > 0,
    segment_summary["customers"] / segment_summary["customers"].sum() * 100,
    np.nan
)
segment_summary["revenue_risk_share_pct"] = np.where(
    segment_summary["historical_revenue_at_risk"].sum() > 0,
    segment_summary["historical_revenue_at_risk"] / segment_summary["historical_revenue_at_risk"].sum() * 100,
    np.nan
)

print_result("Summarized segment customer and revenue exposure by benchmark status.")
display(segment_summary)

Result: Summarized segment customer and revenue exposure by benchmark status.


,benchmark_status,segments,customers,churned_customers,historical_revenue_at_risk,customer_share_pct,revenue_risk_share_pct
0,Above benchmark,1,3226,1516,1944581.19,45.804345,52.777918
1,Below benchmark,2,3817,353,1739878.63,54.195655,47.222082


### Result & conclusion

This summary shows how much of the customer base and historical revenue exposure sits in segments above, within, or below the portfolio churn benchmark.

In [4]:
city_management_view = city_benchmark.copy()
rank_pct = city_management_view["revenue_at_risk"].rank(method="first", pct=True)
city_management_view["exposure_band"] = pd.cut(
    rank_pct,
    bins=[0, 0.25, 0.50, 0.75, 1.000001],
    labels=["Q1 - Lowest exposure", "Q2", "Q3", "Q4 - Highest exposure"],
    include_lowest=True,
)
city_management_view["benchmark_exposure_flag"] = np.select(
    [
        (city_management_view["churn_gap_to_peer_city_pp"] > tolerance_pp) & (city_management_view["exposure_band"] == "Q4 - Highest exposure"),
        (city_management_view["churn_gap_to_peer_city_pp"] > tolerance_pp),
    ],
    ["Above peer benchmark + highest exposure quartile", "Above peer benchmark"],
    default="Other"
)

city_management_view = city_management_view.sort_values(
    ["benchmark_exposure_flag", "revenue_at_risk"], ascending=[True, False]
).reset_index(drop=True)

print_result("Created a neutral management view that combines peer-market churn gaps with revenue-exposure quartiles.")
display(city_management_view.head(30))

Result: Created a neutral management view that combines peer-market churn gaps with revenue-exposure quartiles.


,state,city,customers,churned_customers,avg_cltv,total_revenue,avg_monthly_charge,avg_satisfaction,churn_rate_pct,revenue_at_risk,revenue_at_risk_per_customer,churn_gap_to_portfolio_pp,churn_gap_to_peer_city_pp,cltv_gap_pct,revenue_risk_per_customer_gap_pct,benchmark_status,exposure_band,benchmark_exposure_flag
0,California,Escondido,51,16,4388.86,155899.80,67.88,3.02,31.37,38869.89,762.154706,4.833013,4.75,-0.259886,45.689079,Above peer benchmark,Q3,Above peer benchmark
1,California,Fallbrook,43,26,4531.28,93409.28,69.83,2.72,60.47,34691.78,806.785581,33.933013,33.85,2.976715,54.220459,Above peer benchmark,Q3,Above peer benchmark
2,California,Torrance,25,8,4606.20,77995.90,61.58,2.88,32.00,33534.79,1341.391600,5.463013,5.38,4.679327,156.412649,Above peer benchmark,Q3,Above peer benchmark
3,California,Pasadena,30,9,4226.13,81239.19,63.06,3.10,30.00,26383.57,879.452333,3.463013,3.38,-3.958046,68.111015,Above peer benchmark,Q3,Above peer benchmark
4,California,San Bernardino,28,8,4426.25,88292.72,64.25,3.18,28.57,23726.13,847.361786,2.033013,1.95,0.589830,61.976771,Above peer benchmark,Q3,Above peer benchmark
5,California,Modesto,28,12,4364.64,65847.36,69.56,2.86,42.86,11909.92,425.354286,16.323013,16.24,-0.810304,-18.691738,Above peer benchmark,Q2,Above peer benchmark
6,California,Glendale,40,13,4355.95,90777.55,58.42,3.12,32.50,10622.55,265.563750,5.963013,5.88,-1.007790,-49.236372,Above peer benchmark,Q1 - Lowest exposure,Above peer benchmark
7,California,Berkeley,32,9,4527.31,81479.60,66.71,3.22,28.12,8517.45,266.170313,1.583013,1.50,2.886493,-49.120425,Above peer benchmark,Q1 - Lowest exposure,Above peer benchmark
8,California,San Diego,285,185,4345.09,738416.01,72.02,2.54,64.91,385446.39,1352.443474,38.373013,38.29,-1.254592,158.525262,Above peer benchmark,Q4 - Highest exposure,Above peer benchmark + highest exposure quartile
9,California,San Francisco,104,31,4335.46,306995.99,64.64,3.25,29.81,56894.27,547.060288,3.273013,3.19,-1.473441,4.572876,Above peer benchmark,Q4 - Highest exposure,Above peer benchmark + highest exposure quartile


### Result & conclusion

The management view does not create a subjective score. It simply flags cities where the observed churn rate is above the peer benchmark and shows the relative revenue-exposure quartile.

In [5]:
segment_strategy_view = segment_priority.merge(
    segment_benchmark[[
        "segment_id", "churn_gap_pp", "benchmark_status",
        "revenue_risk_per_customer_gap_pct", "cltv_gap_pct"
    ]],
    on="segment_id",
    how="left",
    validate="one_to_one",
)

segment_strategy_view["priority_share_of_total_pct"] = np.where(
    segment_strategy_view["priority_score"].sum() > 0,
    segment_strategy_view["priority_score"] / segment_strategy_view["priority_score"].sum() * 100,
    np.nan
)

segment_strategy_view = segment_strategy_view.sort_values("priority_score", ascending=False).reset_index(drop=True)

print_result("Connected Phase 7 segment priority scores to Phase 8 benchmark gaps without changing the original priority formula.")
display(segment_strategy_view)

Result: Connected Phase 7 segment priority scores to Phase 8 benchmark gaps without changing the original priority formula.


,segment_id,segment_name,retention_profile,customers,share_pct,avg_tenure_months,avg_monthly_charge,avg_cltv,avg_satisfaction,churned_customers,churn_rate_pct,revenue_at_risk,ease_of_intervention_score,priority_score,priority_basis,priority_order,churn_gap_pp,benchmark_status,revenue_risk_per_customer_gap_pct,cltv_gap_pct,priority_share_of_total_pct
0,0,Segment 0 — Emerging Risk,Retention Priority,3226,45.80,15.74,69.78,4013.45,2.81,1516,46.99,1944581.19,4,7778324.76,Revenue at Risk × Ease of Intervention,1,20.453013,Above benchmark,15.224699,-8.791358,59.449477
1,2,Segment 2 — High-Value At-Risk,Protect High-Value,2260,32.09,57.24,87.39,4972.51,3.45,243,10.75,1696896.61,3,5090689.83,Revenue at Risk × Ease of Intervention,2,-15.786987,Below benchmark,43.526074,13.003995,38.907973
2,1,Segment 1 — High-Value Stable,Maintain Value,1557,22.11,30.80,21.53,4371.24,3.86,110,7.06,42982.02,5,214910.10,Revenue at Risk × Ease of Intervention,3,-19.476987,Below benchmark,-94.723061,-0.660314,1.642551


### Result & conclusion

The combined view lets me read business priority alongside benchmark status. The priority score itself remains the Phase 7 assumption-driven metric, while the benchmark gap remains a separate analytical measure.

In [6]:
external_path = OUTPUT_DIR / "phase_08_external_benchmark_inputs.csv"
external_df = pd.read_csv(external_path) if external_path.exists() else pd.DataFrame()
external_ready = external_df.dropna(subset=["benchmark_value"]).copy() if not external_df.empty else pd.DataFrame()

external_gap = pd.DataFrame(columns=[
    "metric", "benchmark_name", "benchmark_value", "unit", "source", "source_date", "population", "notes", "internal_reference", "gap", "gap_definition"
])

if not external_ready.empty:
    internal_lookup = {
        "churn_rate_pct": float(segment_benchmark["churn_rate_pct"].mul(segment_benchmark["customers"]).sum() / segment_benchmark["customers"].sum()),
    }
    rows = []
    for row in external_ready.itertuples(index=False):
        if row.metric in internal_lookup:
            internal_value = internal_lookup[row.metric]
            rows.append([
                row.metric, row.benchmark_name, float(row.benchmark_value), row.unit, row.source, row.source_date, row.population, row.notes, internal_value, internal_value - float(row.benchmark_value), "Internal portfolio reference minus external benchmark"
            ])
    external_gap = pd.DataFrame(rows, columns=external_gap.columns)

print_result(f"Prepared {len(external_gap)} external benchmark gap records; no external values are required for the internal analysis to remain complete.")
display(external_gap)

Result: Prepared 0 external benchmark gap records; no external values are required for the internal analysis to remain complete.


,metric,benchmark_name,benchmark_value,unit,source,source_date,population,notes,internal_reference,gap,gap_definition


### Result & conclusion

External benchmark gaps are included only when a comparable benchmark value has been explicitly supplied. This prevents unsupported external comparisons from being presented as fact.

In [7]:
benchmark_gap_summary = pd.DataFrame([
    {
        "entity_type": "segment",
        "entities": len(segment_benchmark),
        "above_benchmark": int((segment_benchmark["benchmark_status"] == "Above benchmark").sum()),
        "within_benchmark": int((segment_benchmark["benchmark_status"] == "Within benchmark").sum()),
        "below_benchmark": int((segment_benchmark["benchmark_status"] == "Below benchmark").sum()),
    },
    {
        "entity_type": "state",
        "entities": len(state_benchmark),
        "above_benchmark": int((state_benchmark["benchmark_status"] == "Above benchmark").sum()),
        "within_benchmark": int((state_benchmark["benchmark_status"] == "Within benchmark").sum()),
        "below_benchmark": int((state_benchmark["benchmark_status"] == "Below benchmark").sum()),
    },
    {
        "entity_type": "city",
        "entities": len(city_benchmark),
        "above_benchmark": int((city_benchmark["benchmark_status"] == "Above peer benchmark").sum()),
        "within_benchmark": int((city_benchmark["benchmark_status"] == "Within peer benchmark").sum()),
        "below_benchmark": int((city_benchmark["benchmark_status"] == "Below peer benchmark").sum()),
    },
])

print_result("Created a compact benchmark-status summary across the three analytical grains.")
display(benchmark_gap_summary)

Result: Created a compact benchmark-status summary across the three analytical grains.


,entity_type,entities,above_benchmark,within_benchmark,below_benchmark
0,segment,3,1,0,2
1,state,1,0,1,0
2,city,27,12,3,12


### Result & conclusion

The benchmark summary provides a simple management snapshot without collapsing the underlying details into a single subjective score.

In [8]:
segment_summary.to_csv(OUTPUT_DIR / "segment_benchmark_status_summary.csv", index=False)
city_management_view.to_csv(OUTPUT_DIR / "city_benchmark_management_view.csv", index=False)
segment_strategy_view.to_csv(OUTPUT_DIR / "segment_strategy_benchmark_view.csv", index=False)
external_gap.to_csv(OUTPUT_DIR / "external_benchmark_gap.csv", index=False)
benchmark_gap_summary.to_csv(OUTPUT_DIR / "benchmark_gap_summary.csv", index=False)

print_result("Saved the Phase 8 management views and benchmark summaries.")

Result: Saved the Phase 8 management views and benchmark summaries.


### Result & conclusion

Phase 8 management outputs are now ready for the reporting and MySQL publication layer.